In [ ]:
pip install pandas

In [ ]:
from datetime import datetime

In [ ]:
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []
start_year = datetime.now().year - 5   # last 5 years
end_year = datetime.now().year


for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"⚠️ Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f"⚠️ JSON error for {start_date}: {e}")
            continue

        for f in data["features"]:
            p = f["properties"]
            g = f["geometry"]["coordinates"]
            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),
                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,
                "mag": p.get("mag"),
                "place":p.get("place"),
                "status":p.get("status"),
                "tsunami":p.get("tsunami"),
                "alert":p.get("alert"),
                "felt":p.get("felt"),
                "cdi":p.get("cdi"),
                "mmi":p.get("mni"),
                "sig":p.get("sig"),
               "net":p.get("net"),
              "code":p.get("code"),
              "ids":p.get("ids"),
              "sources":p.get("sources"),
              "types":p.get("types"),
              "nst":p.get("nst"),
              "dmin":p.get("dmin"),
              "rms":p.get("rms"),
              "gap":p.get("gap"),
              "type":p.get("type"),
           })

df = pd.DataFrame(all_records)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.head())


Rows: 113583
Columns: 25
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag  \
0   -68.9337     17.27  4.7   
1  -177.2052    426.71  4.1   
2   121.3159     46.73  4.7   
3    57.2570     10.00  4.9   
4    -3.7578     10.00  4.0   

                                               place    status  tsunami  ...  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed        0  ...   
1                                        Fiji region  reviewed        0  ...   
2                    103 km SW of Basco, Philippines  reviewed        0  ...   
3                

In [ ]:
df.isnull().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
place,0
status,0
tsunami,0


In [ ]:
cols = ['felt', 'cdi', 'mmi', 'nst', 'dmin', 'rms', 'gap']

print(df[cols].skew())

felt    77.441025
cdi      0.735041
mmi           NaN
nst      2.607838
dmin     4.045272
rms      0.147022
gap      0.861549
dtype: object


In [ ]:
print(df.dtypes)

id                   object
time         datetime64[ns]
updated      datetime64[ns]
latitude            float64
longitude           float64
depth_km            float64
mag                 float64
place                object
status               object
tsunami               int64
alert                object
felt                float64
cdi                 float64
mmi                  object
sig                   int64
net                  object
code                 object
ids                  object
sources              object
types                object
nst                 float64
dmin                float64
rms                 float64
gap                 float64
type                 object
dtype: object


In [ ]:
median_cols = ['felt', 'cdi', 'mmi', 'nst', 'dmin', 'gap']

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/tmp/ipykernel_9186/3251200324.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].median())


In [ ]:
df['rms'] = df['rms'].fillna(df['rms'].mean())

In [ ]:
df['alert'] = df['alert'].fillna(df['alert'].mode()[0])

In [ ]:
df.isnull().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
place,0
status,0
tsunami,0


In [ ]:
print(df)

                id                    time                 updated   latitude  \
0       us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040 -31.749300   
1       us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040 -15.490200   
2       us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040  19.752900   
3       us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040  28.152400   
4       us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040  71.321200   
...            ...                     ...                     ...        ...   
113578  us7000spru 2026-06-01 03:34:30.871 2026-06-02 03:57:46.185  30.715800   
113579  us7000sqy4 2026-06-01 02:51:50.510 2026-06-15 05:20:38.040  32.884500   
113580  us7000sqy3 2026-06-01 02:28:03.526 2026-06-15 21:43:32.040  33.046900   
113581  pr71518453 2026-06-01 00:33:28.750 2026-06-01 02:48:31.040  18.909333   
113582  us7000spra 2026-06-01 00:08:41.694 2026-06-08 01:52:41.040 -57.844400   

         longitude  depth_k

In [ ]:
print(df)

                id                    time                 updated   latitude  \
0       us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040 -31.749300   
1       us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040 -15.490200   
2       us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040  19.752900   
3       us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040  28.152400   
4       us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040  71.321200   
...            ...                     ...                     ...        ...   
113122  us7000sps0 2026-06-01 03:58:03.565 2026-06-01 04:17:30.040  30.688400   
113123  us7000sprv 2026-06-01 03:40:11.053 2026-06-01 05:39:48.040 -35.387000   
113124  us7000spru 2026-06-01 03:34:30.871 2026-06-02 03:57:46.185  30.715800   
113125  pr71518453 2026-06-01 00:33:28.750 2026-06-01 02:48:31.040  18.909333   
113126  us7000spra 2026-06-01 00:08:41.694 2026-06-08 01:52:41.040 -57.844400   

         longitude  depth_k

In [ ]:
df.to_csv('earthquake_project.csv', index=False)

from google.colab import files
files.download('earthquake_project.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df=pd.read_csv('/content/earthquake project excel.csv')

In [ ]:
df

,id,time,updated,latitude,longitude,depth_km,mag,place,status,tsunami,...,net,code,ids,sources,types,nst,dmin,rms,gap,type
0,us6000ddi8,20:49.9,02:44.0,-31.749300,-68.933700,17.27,4.70,"29 km SW of Villa Basilio Nievas, Argentina",reviewed,0,...,us,6000ddi8,",us6000ddi8,",",us,",",dyfi,moment-tensor,origin,phase-data,",32,0.2940,0.82,42.0,earthquake
1,us6000dev6,08:17.2,03:47.0,-15.490200,-177.205200,426.71,4.10,Fiji region,reviewed,0,...,us,6000dev6,",us6000dev6,",",us,",",origin,phase-data,",32,1.4710,0.29,64.0,earthquake
2,us6000dev5,54:19.8,03:47.0,19.752900,121.315900,46.73,4.70,"103 km SW of Basco, Philippines",reviewed,0,...,us,6000dev5,",us6000dev5,",",us,",",origin,phase-data,",32,3.0570,0.69,106.0,earthquake
3,us6000ddhs,06:00.8,02:43.0,28.152400,57.257000,10.00,4.90,"114 km N of M?n?b, Iran",reviewed,0,...,us,6000ddhs,",us6000ddhs,",",us,",",origin,phase-data,",32,3.3300,0.61,71.0,earthquake
4,us6000dev4,51:14.0,03:46.0,71.321200,-3.757800,10.00,4.00,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",reviewed,0,...,us,6000dev4,",us6000dev4,",",us,",",origin,phase-data,",32,6.0230,0.50,65.0,earthquake
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113578,us7000spru,34:30.9,57:46.2,30.715800,141.747600,11.00,5.70,"Izu Islands, Japan region",reviewed,0,...,us,7000spru,",usauto7000spru,attfxolo,us7000spru,",",usauto,at,us,",",internal-moment-tensor,internal-origin,losspa...",114,6.0860,0.83,38.0,earthquake
113579,us7000sqy4,51:50.5,20:38.0,32.884500,142.457300,10.00,4.30,"Izu Islands, Japan region",reviewed,0,...,us,7000sqy4,",us7000sqy4,",",us,",",origin,phase-data,",26,5.0220,0.90,141.0,earthquake
113580,us7000sqy3,28:03.5,43:32.0,33.046900,142.546100,10.00,4.40,"off the east coast of Honshu, Japan",reviewed,0,...,us,7000sqy3,",us7000sqy3,",",us,",",origin,phase-data,",31,4.9880,1.08,152.0,earthquake
113581,pr71518453,33:28.7,48:31.0,18.909333,-64.831833,34.17,3.39,"63 km N of Charlotte Amalie, U.S. Virgin Islands",reviewed,0,...,pr,71518453,",us7000sprc,pr71518453,",",us,pr,",",origin,phase-data,",23,0.5263,0.25,236.0,earthquake
